#Reading Data From Bronze

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.types import DateType

In [0]:
#reading sales table from bronze

df_sales = spark.table('workspace.bronze.crm_sales_info')

In [0]:
display(df_sales)

#Transforming Data

Transforming sales table
- trim string values - good practice
- casting Date columns 
- column names are not friendly

## Trimming string values

In [0]:
#trimming all sting values
for field in df_sales.schema.fields:
  if field.dataType == StringType():
      print(field.name)
      df_sales = df_sales.withColumn(field.name, trim(col(field.name)))

In [0]:
display(df_sales)

In [0]:
#validating all the data types. 
for field in df_sales.schema.fields:
  print(field)

## casting Date

In [0]:
#type case the date columns 


df_sales= (
    df_sales
    .withColumn(
        "sls_order_dt",
        when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

In [0]:
display(df_sales)

When date is not having icorrect values 
df_sales = (
    df_sales.withColumn(
    "sls_ship_dt",
    to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    .withColumn(
        "sls_due_dt",
        to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")
    )
)


In [0]:
display(df_sales.limit(5))

## Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df_sales = df_sales.withColumnRenamed(old_name, new_name)

In [0]:
display(df_sales)

In [0]:
df_sales.select("quantity").distinct().show()

##correlating & correcting sales price wrt quantity

In [0]:
df_sales = (
    df_sales
    .withColumn(
        "price",
        when(
            (col("price").isNull()) | (col("price") <= 0),
            when(
                col("quantity") != 0,
                col("sales_amount") / col("quantity")
            ).otherwise(None)
        ).otherwise(col("price"))
    )
)

In [0]:
df_sales.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_sales.columns]).show()

#Writing to Silver 

In [0]:
df_sales.write.mode("overwrite").format('delta').saveAsTable('workspace.silver.crm_sales')

In [0]:
%sql
select * from workspace.silver.crm_sales limit 5;